# 01 — Conhecendo indicadores de saúde: estatística descritiva e visualização

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/01_estatistica_descritiva.ipynb)

**Duração estimada:** 60–75 minutos  
**Pré-requisitos:** Python inicial, pandas e leitura de gráficos.

## Objetivos

- baixar e inspecionar uma base pública
- distinguir variáveis numéricas, binárias e ordinais
- resumir distribuições e escolher gráficos adequados
- comparar pares de variáveis com visualizações e medidas de associação
- discutir autorrelato, associação e limites de generalização

## Fonte e licença

[CDC Diabetes Health Indicators — descrição das variáveis](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators), conjunto 891 da UCI, derivado do BRFSS. Consulte a tabela de variáveis para interpretar os códigos dos atributos, como 0 e 1.

Consulte a licença CC BY 4.0 e a citação exibidas na página da UCI.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Como tornar a execução repetível e sem upload manual?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'ucimlrepo': 'ucimlrepo>=0.0.7,<1', 'scipy': 'scipy>=1.11,<2', 'statsmodels': 'statsmodels>=0.14,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr
from statsmodels.graphics.mosaicplot import mosaic

from src.data_loading import load_cdc_diabetes
from src.visualization import plot_variable

sns.set_theme(style="whitegrid")

## Pergunta orientadora

> Como se distribuem indicadores autorrelatados de saúde nesta amostra, e quais conclusões eles não sustentam?

## Obtenção dos dados

> A fonte e o tamanho da base ficaram registrados?

In [ ]:
data, metadata = load_cdc_diabetes()
print({key: metadata.get(key) for key in ["name", "uci_id", "target", "sample_size"]})
print(metadata["feature_types"])

### Como ler os códigos usados neste encontro

Os números de variáveis categóricas são **rótulos**, não quantidades. Por exemplo, `GenHlth=4` identifica uma categoria e não significa o dobro de `GenHlth=2`. Os nomes legíveis abaixo serão reutilizados nas tabelas, eixos e legendas.

In [ ]:
outcome_labels = {0: "0 Sem diabetes", 1: "1 Pré-diabetes/diabetes"}
high_bp_labels = {0: "0 Sem pressão alta", 1: "1 Com pressão alta"}
health_labels = {
    1: "1 Excelente", 2: "2 Muito boa", 3: "3 Boa",
    4: "4 Regular", 5: "5 Ruim",
}
age_labels = {
    1: "18–24", 2: "25–29", 3: "30–34", 4: "35–39", 5: "40–44",
    6: "45–49", 7: "50–54", 8: "55–59", 9: "60–64", 10: "65–69",
    11: "70–74", 12: "75–79", 13: "80 ou mais",
}

label_dictionary = pd.DataFrame({
    "Variável": ["Diabetes_binary", "HighBP", "GenHlth", "Age"],
    "Significado dos códigos": [
        "; ".join(outcome_labels.values()),
        "; ".join(high_bp_labels.values()),
        "; ".join(health_labels.values()),
        "; ".join(f"{code}={label}" for code, label in age_labels.items()),
    ],
})
display(label_dictionary.style.hide(axis="index"))

## Inspeção

> Qual é o tamanho, o tipo e a qualidade aparente da tabela?

In [ ]:
display(data.head())
print("Dimensão:", data.shape)
display(data.dtypes.rename("tipo").to_frame().T)
display(data.isna().sum().rename("ausências").to_frame().T)
print("Linhas duplicadas:", data.duplicated().sum())

In [ ]:
target_percent = data["Diabetes_binary"].value_counts(normalize=True).sort_index() * 100
target_percent.index = target_percent.index.map(outcome_labels)
display(target_percent.rename("percentual").round(1).to_frame())
ax = target_percent.plot.bar(
    title=f"Indicador de diabetes/pré-diabetes (n={len(data):,})",
    color=["#4C78A8", "#E45756"],
)
ax.set(ylabel="Percentual (%)", xlabel="")
ax.tick_params(axis="x", rotation=0)
plt.show()

### Como interpretar

A barra descreve a prevalência do **indicador-alvo na amostra**. Ela não mede o desempenho de um diagnóstico e não explica por que o desfecho ocorreu.

## Preparação

> Quais resumos são adequados para distribuições possivelmente assimétricas?

In [ ]:
selected = ["BMI", "Age", "GenHlth", "PhysHlth", "MentHlth", "HighBP", "Diabetes_binary"]
description = data[selected].describe(percentiles=[0.25, 0.5, 0.75]).T
description["IQR"] = description["75%"] - description["25%"]
display(description[["count", "mean", "std", "25%", "50%", "75%", "IQR"]].round(2))

## Experimento

> Qual gráfico responde melhor a cada tipo de variável?

In [ ]:
bmi_plot_data = data.assign(**{"Índice de massa corporal (BMI)": data["BMI"]})
plot_variable(bmi_plot_data, "Índice de massa corporal (BMI)")
plt.show()

In [ ]:
high_bp_plot_data = data.assign(
    **{"Pressão arterial elevada": data["HighBP"].map(high_bp_labels)}
)
plot_variable(high_bp_plot_data, "Pressão arterial elevada", kind="categorical")
plt.xticks(rotation=0)
plt.show()

In [ ]:
health_plot_data = data.assign(
    **{"Saúde geral autorrelatada": data["GenHlth"].map(health_labels)}
)
plot_variable(health_plot_data, "Saúde geral autorrelatada", kind="ordinal")
plt.xticks(rotation=25, ha="right")
plt.show()

### Como interpretar

Para BMI, mediana e IQR são resistentes a extremos. Para variáveis binárias e ordinais, percentuais preservam uma leitura mais direta. Diferenças entre grupos são associações descritivas, não efeitos causais.

## Avaliação visual conjunta

> Que relações aparecem entre os indicadores?

In [ ]:
pair_frame = pd.DataFrame({
    "IMC": data["BMI"],
    "Saúde física ruim (dias/30)": data["PhysHlth"],
    "Saúde mental ruim (dias/30)": data["MentHlth"],
    "Indicador de diabetes": data["Diabetes_binary"].map(outcome_labels),
})
sns.pairplot(
    pair_frame, hue="Indicador de diabetes", corner=True,
    plot_kws={"alpha": 0.35, "s": 18},
)
plt.show()

### Como interpretar

Os eixos contêm apenas medidas numéricas; a categoria aparece na legenda com seu significado. Sobreposição indica que um atributo isolado dificilmente separa perfeitamente os grupos.

## Comparações bivariadas adequadas ao tipo

O `pairplot` é útil para variáveis numéricas, mas códigos de categorias não devem ser interpretados como medidas contínuas. As estratégias seguintes preservam a natureza das variáveis.

### Duas variáveis categóricas

Vamos comparar a saúde geral autorrelatada (`GenHlth`) com o indicador binário `Diabetes_binary`. Percentuais condicionais evitam que o tamanho desigual das categorias domine a leitura.

In [ ]:
health_order = list(health_labels.values())
outcome_order = list(outcome_labels.values())

comparison = data.assign(
    GenHlth_label=data["GenHlth"].map(health_labels),
    outcome_label=data["Diabetes_binary"].map(outcome_labels),
)
comparison["GenHlth_label"] = pd.Categorical(
    comparison["GenHlth_label"], categories=health_order, ordered=True
)
comparison["outcome_label"] = pd.Categorical(
    comparison["outcome_label"], categories=outcome_order, ordered=True
)

contingency = pd.crosstab(
    comparison["GenHlth_label"], comparison["outcome_label"], dropna=False
)
row_percent = contingency.div(contingency.sum(axis=1), axis=0) * 100
display(contingency.rename_axis("Saúde geral"))
display(row_percent.rename_axis("Saúde geral").round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
sns.heatmap(
    row_percent, annot=True, fmt=".1f", cmap="Blues", ax=axes[0],
    cbar_kws={"label": "% dentro da categoria de saúde geral"},
)
axes[0].set_title("Heatmap de percentuais condicionais")
axes[0].set_xlabel("Indicador-alvo")
axes[0].set_ylabel("Saúde geral autorrelatada")

row_percent.plot.bar(
    stacked=True, ax=axes[1], color=["#4C78A8", "#E45756"], width=0.8
)
axes[1].set(title="Barras 100% empilhadas", xlabel="Saúde geral autorrelatada", ylabel="Percentual (%)")
axes[1].set_ylim(0, 100)
axes[1].legend(title="Indicador-alvo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

In [ ]:
outcome_by_health = (
    comparison.groupby("GenHlth_label", observed=False)["Diabetes_binary"]
    .agg(n="size", proporcao="mean")
)
outcome_by_health["percentual_desfecho"] = 100 * outcome_by_health["proporcao"]
display(outcome_by_health[["n", "percentual_desfecho"]].round(1))

ax = outcome_by_health["percentual_desfecho"].plot.bar(
    color="#E45756", figsize=(8, 4), title="Proporção do desfecho por categoria"
)
ax.set(xlabel="Saúde geral autorrelatada", ylabel="Pré-diabetes/diabetes (%)")
ax.set_ylim(0, max(5, outcome_by_health["percentual_desfecho"].max() * 1.12))
plt.show()

In [ ]:
mosaic_data = comparison.rename(columns={
    "GenHlth_label": "Saúde geral",
    "outcome_label": "Indicador de diabetes",
})
fig, _ = mosaic(
    mosaic_data.dropna(subset=["Saúde geral", "Indicador de diabetes"]),
    ["Saúde geral", "Indicador de diabetes"],
    title="Mosaico: saúde geral × indicador de diabetes",
)
fig.set_size_inches(11, 6)
plt.show()

In [ ]:
chi2, p_value, degrees_freedom, expected = chi2_contingency(contingency)
n_observations = contingency.to_numpy().sum()
min_dimension = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
cramers_v = (chi2 / (n_observations * min_dimension)) ** 0.5

categorical_association = pd.Series({
    "qui_quadrado": chi2,
    "graus_de_liberdade": degrees_freedom,
    "p_valor": p_value,
    "V_de_Cramer": cramers_v,
    "menor_frequencia_esperada": expected.min(),
})
display(categorical_association.to_frame("valor").round(4))

O heatmap e as barras usam percentuais **dentro de cada categoria de saúde geral**. O mosaico acrescenta o tamanho relativo das categorias pela área dos retângulos. O teste qui-quadrado avalia incompatibilidade com independência; o V de Cramér resume a magnitude da associação entre 0 e 1. Com amostras grandes, um p-valor pequeno pode acompanhar uma associação fraca, por isso gráficos e tamanho de efeito devem ser examinados juntos.

### Uma variável categórica e uma numérica

Agora comparamos o indicador categórico `Diabetes_binary` e a variável numérica `BMI`. Resumos por grupo e gráficos de distribuição são mais informativos do que uma nuvem de pontos com categorias codificadas como números.

In [ ]:
bmi_summary = (
    comparison.groupby("outcome_label", observed=False)["BMI"]
    .agg(
        n="size", media="mean", mediana="median", desvio_padrao="std",
        q1=lambda values: values.quantile(0.25),
        q3=lambda values: values.quantile(0.75),
    )
)
bmi_summary["IQR"] = bmi_summary["q3"] - bmi_summary["q1"]
display(bmi_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.boxplot(data=comparison, x="outcome_label", y="BMI", order=outcome_order, ax=axes[0])
axes[0].set(title="Boxplot por grupo", xlabel="Indicador-alvo", ylabel="BMI")
sns.violinplot(
    data=comparison, x="outcome_label", y="BMI", order=outcome_order,
    inner="quartile", cut=0, ax=axes[1],
)
axes[1].set(title="Distribuição e quartis", xlabel="Indicador-alvo", ylabel="BMI")
plt.show()

In [ ]:
valid_bmi = comparison[["Diabetes_binary", "BMI"]].dropna()
r_point_biserial, p_point_biserial = pointbiserialr(
    valid_bmi["Diabetes_binary"], valid_bmi["BMI"]
)
grouped_bmi = valid_bmi.groupby("Diabetes_binary")["BMI"].agg(["size", "mean", "var"])
pooled_variance = (
    ((grouped_bmi["size"] - 1) * grouped_bmi["var"]).sum()
    / (grouped_bmi["size"].sum() - len(grouped_bmi))
)
standardized_mean_difference = (
    (grouped_bmi.loc[1, "mean"] - grouped_bmi.loc[0, "mean"])
    / pooled_variance ** 0.5
)

numeric_association = pd.Series({
    "diferenca_das_medias_BMI": grouped_bmi.loc[1, "mean"] - grouped_bmi.loc[0, "mean"],
    "r_ponto_bisserial": r_point_biserial,
    "p_valor_r": p_point_biserial,
    "eta_quadrado": r_point_biserial ** 2,
    "diferenca_padronizada_medias": standardized_mean_difference,
})
display(numeric_association.to_frame("valor").round(4))

A correlação ponto-bisserial quantifica a associação entre uma categoria binária e uma medida numérica; η² é seu quadrado neste caso e resume a fração da variabilidade amostral situada entre os dois grupos. A diferença padronizada de médias facilita a comparação de magnitude na escala do desvio-padrão. Sinal, p-valor e tamanho de efeito dependem da codificação e não controlam confundimento nem demonstram causalidade.

## Limitações e responsabilidade

- Os indicadores do BRFSS incluem autorrelato, sujeito a memória e classificação incorreta.
- Amostragem, representação de grupos e desbalanceamento limitam a generalização.
- Uma associação visual não estabelece causalidade nem substitui avaliação clínica.

## Atividade

Escolha uma variável categórica diferente de `GenHlth` e repita a comparação com o desfecho. Depois escolha uma variável numérica diferente de `BMI`, compare os grupos e relate: (1) percentuais condicionais; (2) tamanho de efeito; (3) o que não pode ser concluído.

## Três aprendizados principais

1. O tipo da variável orienta o resumo e o gráfico.
2. Percentuais condicionais e distribuições por grupo evitam tratar códigos como medidas contínuas.
3. Gráficos, testes e tamanhos de efeito descrevem associação, mas não provam causalidade.

## Referências

- [UCI — CDC Diabetes Health Indicators](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)
- [CDC — BRFSS](https://www.cdc.gov/brfss/)
- [statsmodels — mosaic plot](https://www.statsmodels.org/stable/generated/statsmodels.graphics.mosaicplot.mosaic.html)
- [SciPy — chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.chi2_contingency.html)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy', 'scikit-learn', 'statsmodels', 'ucimlrepo'))